In [2]:
import re
import requests
from bs4 import BeautifulSoup
from supabase import create_client
import os

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
TRAINER_WEB_USERNAME = os.getenv("TRAINER_WEB_USERNAME")
TRAINER_WEB_PASSWORD = os.getenv("TRAINER_WEB_PASSWORD")

DIVISIONS = ["master", "senior", "junior"]
TRAINER_ID_PATTERN = re.compile(r"^ph\d{8}$")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
session = requests.Session()

headers = {
    "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "content-type": "application/x-www-form-urlencoded",
    "origin": "https://asia.pokemon-card.com",
    "referer": "https://asia.pokemon-card.com/ph/login/",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
}

login_payload = {
    "redirectTo": "",
    "email": TRAINER_WEB_USERNAME,
    "password": TRAINER_WEB_PASSWORD,
}

login_res = session.post(
    "https://asia.pokemon-card.com/ph/login/", headers=headers, data=login_payload
)

if login_res.status_code != 200:
    print(f"❌ Login failed with status: {login_res.status_code}")
    exit(1)

print("✅ Login successful (or session established)")

trainers_data = []
scraped_ids = set()

for division in DIVISIONS:
    ranking_url = f"https://asia.pokemon-card.com/ph/mypage/ranking/{division}/"
    res = session.get(ranking_url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")

    for row in soup.select(".userInformation"):
        text = row.get_text(separator=" ", strip=True)
        data = text.split(" ")

        trainer_id = data.pop()
        nickname = data.pop()

        if not TRAINER_ID_PATTERN.match(trainer_id):
            print(f"⚠️ Skipping unexpected ID format: {trainer_id!r}")
            continue

        scraped_ids.add(trainer_id)
        trainers_data.append({
            "id": trainer_id,
            "nickname": nickname,
            "division": division,
            "full_name": None,
            "alphabet_name": None,
            "is_deleted": False,
        })

if not trainers_data:
    print("⚠️ No trainer data found. Check if selectors need updating.")
    exit(0)

# Diff: fetch existing IDs and mark deleted ones BEFORE upserting
try:
    existing = (
        supabase.table("trainers")
        .select("id")
        .eq("is_deleted", False)
        .execute()
    )
    existing_ids = {row["id"] for row in existing.data}
    deleted_ids = existing_ids - scraped_ids

    if deleted_ids:
        supabase.table("trainers").update({"is_deleted": True}).in_(
            "id", list(deleted_ids)
        ).execute()
        print(f"🗑️  Marked {len(deleted_ids)} trainer(s) as deleted: {deleted_ids}")
    else:
        print("✅ No deleted trainers detected.")
except Exception as e:
    print(f"❌ Supabase Error (deletion check): {e}")

# Upsert new/updated trainer data
print(trainers_data)
try:
    supabase.table("trainers").upsert(trainers_data).execute()
    print(f"🚀 Successfully synced {len(trainers_data)} trainers to Supabase.")
except Exception as e:
    print(f"❌ Supabase Error (upsert): {e}")
    exit(1)

✅ Login successful (or session established)
✅ No deleted trainers detected.
[{'id': 'ph35909376', 'nickname': 'Bert', 'division': 'master', 'full_name': None, 'alphabet_name': None, 'is_deleted': False}, {'id': 'ph63343087', 'nickname': 'JJCBN', 'division': 'master', 'full_name': None, 'alphabet_name': None, 'is_deleted': False}, {'id': 'ph00891794', 'nickname': 'Gid', 'division': 'master', 'full_name': None, 'alphabet_name': None, 'is_deleted': False}, {'id': 'ph95280670', 'nickname': 'Sky', 'division': 'master', 'full_name': None, 'alphabet_name': None, 'is_deleted': False}, {'id': 'ph71254123', 'nickname': 'Gladshark', 'division': 'master', 'full_name': None, 'alphabet_name': None, 'is_deleted': False}, {'id': 'ph84249065', 'nickname': 'Aaron', 'division': 'master', 'full_name': None, 'alphabet_name': None, 'is_deleted': False}, {'id': 'ph91897557', 'nickname': '[HS]Leo', 'division': 'master', 'full_name': None, 'alphabet_name': None, 'is_deleted': False}, {'id': 'ph23472153', 'nick